# Query Pipeline — COSORA

Busqueda hibrida (Dense + BM25 con RRF), prompt builder y generacion con LLM.

**Prerequisito:** ejecutar `ingestion_pipeline.ipynb` primero.

## 0. Setup

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
!pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv


In [29]:
import chromadb

client = chromadb.PersistentClient(
    path="/drive/MyDrive/RAG_UPC_Final_project/chroma_db"
)

# Obtenemos la lista
colecciones = client.list_collections()

for col_item in colecciones:
    # Si col_item es solo texto (string), obtenemos el objeto de la colección
    if isinstance(col_item, str):
        col = client.get_collection(name=col_item)
    else:
        # Si ya es un objeto, lo usamos tal cual
        col = col_item
        
    print(f"📦 '{col.name}' → {col.count()} chunks")


In [30]:

import os
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import torch
from transformers import pipeline
from dotenv import load_dotenv

# Cargar variables de entorno desde Drive
env_path = '/content/drive/MyDrive/variablentorno/.env'
if os.path.exists(env_path):
    load_dotenv(env_path)
    print('✅ Variables de entorno cargadas')

✅ Variables de entorno cargadas


In [31]:
# Config
CHROMA_PATH     = '/content/drive/MyDrive/RAG_UPC_Final_project/chroma_db'
COLLECTION_NAME = 'cosora_chunks'
EMBED_MODEL     = 'intfloat/multilingual-e5-base'

# Backend de generacion: "local" o "openai"
GEN_BACKEND = 'openai'
LOCAL_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# Conectar
client_chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = client_chroma.get_collection(COLLECTION_NAME)
embed_model   = SentenceTransformer(EMBED_MODEL)

print(f'ChromaDB: {collection.count()} chunks')
print(f'Backend: {GEN_BACKEND}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB: 399 chunks
Backend: openai


## 1. Retrieval hibrido

In [32]:
def dense_search(query, k=20):
    q_vec = embed_model.encode(f"query: {query}").tolist()
    res   = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {'text': doc, 'meta': meta, 'rank_dense': i}
        for i, (doc, meta) in enumerate(zip(res['documents'][0], res['metadatas'][0]))
    ]

In [33]:
import re

# Stopwords basicos en español para limpiar ruido en BM25
STOPWORDS_ES = set([
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 
    'para', 'con', 'no', 'una', 'su', 'al', 'es', 'lo', 'como', 'más', 'pero', 'sus', 
    'le', 'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 
    'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 
    'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso', 'ante', 
    'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 
    'otra', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada', 'muchos', 'cual', 
    'poco', 'ella', 'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tú', 
    'te', 'ti', 'tu', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os', 'mío', 
    'mía', 'míos', 'mías', 'tuyo', 'tuya', 'tuyos', 'tuyas', 'suyo', 'suya', 'suyos', 
    'suyas', 'nuestro', 'nuestra', 'nuestros', 'nuestras', 'vuestro', 'vuestra', 'vuestros', 
    'vuestras', 'esos', 'esas', 'aquel', 'aquella', 'aquellos', 'aquellas', 'cuales', 
    'cuanto', 'cuanta', 'cuantos', 'cuantas', 'cuan'
])

def tokenize_bm25(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Eliminar puntuación
    return [word for word in text.split() if word and word not in STOPWORDS_ES]

# Indice BM25 sobre todos los chunks
all_data  = collection.get()
all_docs  = all_data['documents']
all_metas = all_data['metadatas']

tokenized  = [tokenize_bm25(doc) for doc in all_docs]
bm25_index = BM25Okapi(tokenized)
print(f'Indice BM25: {len(all_docs)} chunks')

def bm25_search(query, k=20):
    scores  = bm25_index.get_scores(tokenize_bm25(query))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {'text': all_docs[i], 'meta': all_metas[i], 'rank_bm25': rank}
        for rank, i in enumerate(top_idx)
    ]


Indice BM25: 399 chunks


In [34]:
def rrf_fusion(dense, bm25, k=60, top_n=5):
    """Reciprocal Rank Fusion."""
    scores = {}

    for item in dense:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_dense'])

    for item in bm25:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_bm25'])

    ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
    return ranked[:top_n]

## 2. Prompt Builder

In [35]:
def build_prompt(query, chunks):
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        doc_id = chunk['meta']['doc_id']
        context_blocks.append(f'[Fragmento {i} - Fuente: {doc_id}]\n{chunk["text"]}')

    context = '\n\n'.join(context_blocks)

    return f"""Eres COSORA, un asistente experto en obra ferroviaria. \
Responde UNICAMENTE con la informacion del contexto proporcionado. \
Si la informacion no esta en el contexto, dilo explicitamente. \
Cita siempre la fuente (nombre del acta). Responde en espanol.

=== CONTEXTO ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""

## 3. Generador LLM

In [36]:
if GEN_BACKEND == 'local':
    print(f'Cargando {LOCAL_MODEL}...')
    generator = pipeline(
        'text-generation',
        model=LOCAL_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    print('Modelo cargado')


def generate_answer(prompt):
    if GEN_BACKEND == 'local':
        output = generator(
            prompt,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        return output[0]['generated_text'][len(prompt):].strip()

    elif GEN_BACKEND == 'openai':
        from openai import OpenAI
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            raise ValueError("No se encontro OPENAI_API_KEY en el .env. Comprueba la ruta en la celda de Setup.")
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content

    raise ValueError(f'Backend desconocido: {GEN_BACKEND}')

## 4. Pipeline completo

In [37]:
def ask_cosora(query, verbose=True):
    dense  = dense_search(query, k=20)
    bm25   = bm25_search(query, k=20)
    chunks = rrf_fusion(dense, bm25, top_n=20)

    # 1. Verificación de Relevancia (Score Thresholding)
    if not chunks or chunks[0]['score'] < 0.015:
        msg = "⚠️ Aviso: No se encontró información suficientemente relevante en las actas para responder a esta pregunta."
        if verbose:
            print(f'Query: {query}')
            print(msg)
            print('-' * 70)
        return msg

    # 2. Limpieza del prefijo 'passage: ' para OpenAI
    for c in chunks:
        c['text'] = c['text'].replace('passage: ', '', 1)

    prompt = build_prompt(query, chunks)
    answer = generate_answer(prompt)

    if verbose:
        print(f'Query: {query}')
        print(f'\nChunks recuperados ({len(chunks)}):')
        for c in chunks:
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f} | {c['text'][:80]}...")
        print(f'\nRespuesta ({GEN_BACKEND}):\n{answer}')
        print('-' * 70)


## 5. Tests

In [38]:
queries_test = [
    'Que se decidio sobre el talud?',
    'Cual es el estado del camino provisional?',
    'Que materiales se aprobaron para el drenaje de muros?',
]

for q in queries_test:
    ask_cosora(q)
    print()

Query: Que se decidio sobre el talud?

Chunks recuperados (20):
  - [254275-DO-AVO-14-V07] score=0.0328 | contra incendios. De esta manera tener una última revisión y certeza de lo que s...
  - [254275-DO-AVO-15-V07] score=0.0328 | los informes de revisión del PC y PGMA. AR-29 Estabilidad del talud Posterior a ...
  - [254275-DO-AVO-16-V07] score=0.0323 | camino provisional DF solicita a la UTE que aporte la documentación gráfica que ...
  - [254275-DO-AVO-15-V07] score=0.0318 | de la gestión. AR-14 Instalaciones CSIC DO-DEO-UTE convienen en la necesidad de ...
  - [254275-DO-AVO-22-V01] score=0.0291 | andén. Se revisan los siguientes aspectos mostrados en el plano: Acceso provisio...
  - [254275-DO-AVO-22-V01] score=0.0159 | solera en voladizo, situado en la parte superior del talud de excavación, que ac...
  - [254275-DO-AVO-17-V07] score=0.0156 | una capa de acabado de cinta de polietileno. Descripción de protección también i...
  - [244170-DOB-AVO-18-V00-A0] score=0.0154 | |inspecc

In [39]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

query = "Que se decidio sobre el talud?"
doc_relevante = "3.5. Estabilidad del talud. UTE proporciona un plano..."
doc_irrelevante = "El pavimento definido en la partida de presupuesto..."

for name in ["BSC-LT/MrBERT-es", "intfloat/multilingual-e5-base"]:
    model = SentenceTransformer(name)
    
    # Añadimos los prefijos obligatorios solo para la familia e5
    if "e5" in name:
        q = model.encode(f"query: {query}")
        r = model.encode(f"passage: {doc_relevante}")
        i = model.encode(f"passage: {doc_irrelevante}")
    else:
        q = model.encode(query)
        r = model.encode(doc_relevante)
        i = model.encode(doc_irrelevante)
    
    sim_rel = cosine_similarity([q], [r])[0][0]
    sim_irr = cosine_similarity([q], [i])[0][0]
    print(f"{name}:")
    print(f"  Similaridad con doc relevante:   {sim_rel:.4f}")
    print(f"  Similaridad con doc irrelevante: {sim_irr:.4f}")

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BSC-LT/MrBERT-es:
  Similaridad con doc relevante:   0.9160
  Similaridad con doc irrelevante: 0.9204


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


intfloat/multilingual-e5-base:
  Similaridad con doc relevante:   0.8128
  Similaridad con doc irrelevante: 0.7689
